# Load Data from Bronze and Silver Layers

This step loads the data from the Bronze and Silver tables into separate Spark DataFrames.

The Bronze data will be used as a reference for data validation, including record count consistency checks.

The Silver data will be used for data validation and as the source for all subsequent transformations required to build the dimension and fact tables in the Gold layer.

In [0]:
# Define source silver and bronze table
SILVER_TABLE = "workspace.weather_data.weather_silver"
BRONZE_TABLE = "workspace.weather_data.weather_raw"
# Load transformed data from bronze and silver tables
silver_df = spark.read.table(SILVER_TABLE)
bronze_df = spark.read.table(BRONZE_TABLE)

# Display schemas and few records from both tables
print("Bronze table schema:")
bronze_df.printSchema()
print("Silver table schema:")
silver_df.printSchema()

print("\nBronze table records:")
display(bronze_df.limit(10))
print("\nSilver table records:")
display(silver_df.limit(10))

%md
# Validate Record Count Consistency Between Bronze and Silver Layers

This step compares the total number of records stored in the Bronze and Silver layers.

The Silver layer applies data transformations without adding or removing records.

The record counts must match between both layers.

The pipeline stops if a difference in record counts is detected.

In [0]:
# Counting rows in Bronze and Silver Tables
rows_bronze = bronze_df.count()
rows_silver = silver_df.count()
# Check if the number of rows in the silver table is equal to the number of rows in the bronze table
if rows_bronze != rows_silver:
   raise ValueError(f"Data quality check failed: Record count in bronze table: {rows_bronze}; Record count in silver table: {rows_silver}")
else:
    print(f"Data quality check passed: There are {rows_bronze} rows in both tables")

# Create the City Dimension Table

Starting from this step, SQL is used to perform data transformations and create dimension and fact tables in the Gold layer.

This approach separates data ingestion and preparation, handled with PySpark, from analytical data modeling, implemented using SQL.

The `dim_city` dimension table is created using data from the Silver layer.

The transformation consists of the following operations:

- Extract unique combinations of city names and geographic coordinates.
- Generate a unique surrogate key (`city_id`) for each city using the `ROW_NUMBER()` window function.
- Store the resulting dataset as a Delta table.

The `dim_city` table contains one record per city and serves as a reference dimension for the `fact_weather` table.

The table is created using `CREATE OR REPLACE TABLE`, allowing the transformation to be rerun and the dimension table to be fully refreshed.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.weather_data.dim_city
  USING DELTA AS
SELECT
  -- Adding a surrogate key to the dim_city table
  ROW_NUMBER() OVER (ORDER BY city) AS city_id,
  *
FROM
  (
    -- Using subquery to select distinct city, latitude, longitude
    SELECT
    DISTINCT
      city,
      latitude,
      longitude
    FROM
      workspace.weather_data.weather_silver
  )

# Create the Date Dimension Table

This step creates the `dim_date` dimension table in the Gold layer using a dynamically generated calendar.

The calendar is generated independently of the individual records in the Silver layer. The Silver table is used only to determine the latest available date.

The transformation consists of the following operations:

- Define a fixed calendar start date of January 1, 2020.
- Retrieve the maximum available date from the Silver layer to determine the calendar end date.
- Generate a continuous sequence of calendar dates using the `SEQUENCE` and `EXPLODE` functions.
- Generate a surrogate key (`date_key`) for each date using the `yyyyMMdd` format.
- Derive calendar attributes, including day, month, month name, quarter, year, day of the week, and weekend indicator.
- Store the resulting dataset as a Delta table.

The `dim_date` table contains one record per calendar day, including dates without corresponding weather measurements.

The dynamic end date supports future incremental loading by allowing the calendar to expand as new dates become available in the Silver layer.

The initial table is created using `CREATE OR REPLACE TABLE`. Incremental loading will be implemented as a separate process in a subsequent development step.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.weather_data.dim_date
  USING DELTA AS
-- Create a table with all dates between 2020-01-01 and the most recent date in the silver table
WITH dates AS (
  SELECT
    EXPLODE(SEQUENCE(DATE "2020-01-01", MAX(DATE), INTERVAL 1 DAY)) AS full_date
  FROM
    workspace.weather_data.weather_silver
)
--Adding additional columns for date_key, day, month, quarter, year, day_of_week, is_weekend
--Note: 0 = Sunday, 1 = Monday
SELECT
  CAST(DATE_FORMAT(full_date, "yyyyMMdd") AS INT) AS date_key,
  full_date,
  DAY(full_date) AS day,
  MONTH(full_date) AS month,
  MONTHNAME(full_date) AS month_name,
  QUARTER(full_date) AS quarter,
  YEAR(full_date) AS year,
  WEEKDAY(full_date) + 1 AS day_of_week,
  CASE
    WHEN (weekday(full_date) + 1) IN (6, 7) THEN TRUE
    ELSE FALSE
  END AS is_weekend
FROM
  dates

# Create the Weather Fact Table

This step creates the `fact_weather` fact table in the Gold layer using data from the Silver layer and the dimension tables.

The transformation consists of the following operations:

- Join the Silver table with `dim_city` using the city name.
- Join the Silver table with `dim_date` using the `date` column from Silver and the `full_date` column from `dim_date`.
- Retrieve the corresponding surrogate keys (`city_id` and `date_key`).
- Select the hourly weather measurements required for analytical reporting.
- Store the resulting dataset as a Delta table.

The `fact_weather` table uses an hourly grain, with one record per city and measurement timestamp.

The table contains foreign keys referencing `dim_city` and `dim_date`, the original measurement timestamp, and weather measurements.

The table is created using `CREATE OR REPLACE TABLE`, allowing the transformation to be rerun and the fact table to be fully refreshed.

In [0]:
%sql
--Create Fact Table
CREATE OR REPLACE TABLE workspace.weather_data.fact_weather
USING DELTA
SELECT
  dc.city_id,
  dd.date_key,
  ws.measurement_time,
  ws.temperature_2m,
  ws.apparent_temperature,
  ws.relative_humidity_2m,
  ws.precipitation,
  ws.rain,
  ws.snowfall,
  ws.cloud_cover,
  ws.wind_speed_10m,
  ws.wind_gusts_10m,
  ws.wind_direction_10m,
  ws.shortwave_radiation
FROM
  workspace.weather_data.weather_silver ws
  --Joining silver table with dim tables to aquire surrogate keys
    LEFT JOIN workspace.weather_data.dim_city dc
      ON dc.city = ws.city
    LEFT JOIN workspace.weather_data.dim_date dd
      ON dd.full_date = ws.date

# Validate Record Count Consistency Between Silver and Gold Layers

This step compares the total number of records stored in the Silver layer with the total number of records in the Gold fact table.

The `fact_weather` table preserves the hourly data grain from the Silver layer without aggregating or removing records.

The record counts must match between both layers.



In [0]:
# Establishing dataframe from fact_weather table
fact_weather_df = spark.read.table("workspace.weather_data.fact_weather")

# Counting rows in fact_weather and weather_silver tables
fact_count = fact_weather_df.count()
silver_count = silver_df.count()

# Check if the number of rows in the silver table is equal to the number of rows in the fact_weather table
if fact_count == silver_count:
    print(f"Data quality check passed: There are {silver_count} rows in both tables")
else:
    raise ValueError(
        f"Data quality check failed: Record count in silver table: {silver_count}; Record count in fact_weather table: {fact_count}"
    )

# Validate Dimension Key Uniqueness

This step verifies that the surrogate keys in the dimension tables are unique.

The validation consists of the following checks:

- Compare the total number of records in `dim_date` with the number of distinct `date_key` values.
- Compare the total number of records in `dim_city` with the number of distinct `city_id` values.

Each dimension key must uniquely identify one record in its respective dimension table.

The pipeline stops if duplicate dimension keys are detected.

In [0]:
# Establish dataframe from dim_date table
dim_date = spark.read.table("workspace.weather_data.dim_date")

# Count rows in dim_date table and the number of unique date_key values
dim_date_count = dim_date.count()
unique_date_key_count = dim_date.select("date_key").distinct().count()

# Check if the number of rows in the dim_date table is equal to the number of unique date_key values
if dim_date_count == unique_date_key_count:
    print(f"Data quality check passed: Every date in dim_date has a unique date_key")
else:
    raise ValueError(
        f"Data quality check failed: There are {dim_date_count} rows in dim_date, but only {unique_date_key_count} unique date_key values"
    )

# Establish dataframe from dim_city table
dim_city = spark.read.table("workspace.weather_data.dim_city")

# Count rows in dim_city table and the number of unique city_id values
dim_city_count = dim_city.count()
unique_city_id_count = dim_city.select("city_id").distinct().count()

# Check if the number of rows in the dim_city table is equal to the number of unique city_id values
if dim_city_count == unique_city_id_count:
    print(f"Data quality check passed: Every city in dim_city has a unique city_id")
else:
    raise ValueError(
        f"Data quality check failed: There are {dim_city_count} rows in dim_city, but only {unique_city_id_count} unique city_id values"
    )